# NB01 — download + fixed preamble + feature cache, shard 2/3 (CPU)

Subjects `sorted(participants)[2::3]`. Downloads only this shard's raw `.set` files from OpenNeuro S3, runs the fixed preamble (filter, notch, avg-ref, ICA+ICLabel, 128 Hz), then builds P1–P7 features, wPLI and PSWE outputs. Shards are merged in NB02.

In [ ]:
REPO = 'https://github.com/Devguru-codes/Major_project_IIITN.git'
BRANCH = 'feat/implementation'
WORK = '/kaggle/working'
SRC = '/tmp/eegrep-src'   # code checkout stays out of the saved output

import subprocess, sys, os, json, platform, shutil

def sh(cmd, log=None):
    print('$', cmd, flush=True)
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    print(r.stdout[-20000:], r.stderr[-5000:], sep='\n', flush=True)
    if log:
        open(log, 'w').write(r.stdout + '\n' + r.stderr)
    if r.returncode != 0:
        raise RuntimeError(f'command failed ({r.returncode}): {cmd}')
    return r.stdout

shutil.rmtree(SRC, ignore_errors=True)
sh(f'git clone --depth 1 -b {BRANCH} {REPO} {SRC}')
SHA = sh(f'git -C {SRC} rev-parse --short HEAD').strip()
sh(f'pip install -q -e "{SRC}[preprocess]"', log=f'{WORK}/pip_install.log')
PY = f'cd {SRC} && {sys.executable} -m eegrep'
print('git sha', SHA)

In [ ]:
SHARD, N_SHARDS = 2, 3
sh(f'{PY} download --dest /tmp/ds004504 --shard {SHARD} --n-shards {N_SHARDS}', log=f'{WORK}/download.log')
shutil.copy('/tmp/ds004504/download_manifest.json', f'{WORK}/download_manifest_s{SHARD}.json')
shutil.copy('/tmp/ds004504/participants.tsv', f'{WORK}/participants.tsv')

In [ ]:
sh(f'{PY} preprocess --bids /tmp/ds004504 --out {WORK}/preproc --shard {SHARD} --n-shards {N_SHARDS} --n-jobs 4', log=f'{WORK}/preprocess.log')

In [ ]:
sh(f'{PY} cache --preproc {WORK}/preproc --participants {WORK}/participants.tsv --out {WORK}/cache_s{SHARD}', log=f'{WORK}/cache.log')

In [ ]:
import torch
sh(f'{sys.executable} -m pip freeze', log=f'{WORK}/pip_freeze.txt')
env = {'git_sha': SHA, 'python': platform.python_version(), 'torch': torch.__version__,
       'cuda': torch.cuda.is_available(), 'cpu_count': os.cpu_count(),
       'kaggle_image': os.environ.get('KAGGLE_DOCKER_IMAGE', 'unknown')}
json.dump(env, open(f'{WORK}/environment.json', 'w'), indent=1)
print(env)